In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Fine-tuning Qwen for **InvestIQ**

In this notebook, we fine-tune the `Qwen/Qwen2.5-1.5B-Instruct` language model for the InvestIQ project.

The goal is to adapt the model to respond to investment-related questions using the structured response format defined for InvestIQ. We will use **LoRA (Low-Rank Adaptation)** so that we can fine-tune the model efficiently without updating all of its parameters.

The fine-tuning workflow is:

1. Install the required libraries
2. Connect Google Drive and locate the training dataset
3. Load and inspect the Qwen tokenizer
4. Inspect the chat template and training format
5. Load the Qwen model
6. Configure LoRA
7. Prepare the training dataset
8. Fine-tune the model
9. Evaluate the fine-tuned model

# 1. Install Required Libraries

Before fine-tuning the model, we prepare the Google Colab environment and authenticate with Hugging Face.

The fine-tuning workflow uses several Python libraries.

* **Transformers** provides the Qwen tokenizer and language model.
* **Datasets** is used to load and prepare the JSONL training dataset.
* **PEFT** provides LoRA for parameter-efficient fine-tuning.
* **Accelerate** helps manage the training process and hardware.
* **PyTorch** is the deep learning framework used by Qwen. Google Colab already provides PyTorch, so we will use the installed version rather than reinstalling it.

In [3]:
!pip install -q -U torchao transformers datasets peft accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 29.6 MB/s eta 0:00:00


In [4]:
# Check CPU memory
!free -h
!nvidia-smi

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.2Gi       7.1Gi       2.4Mi       4.7Gi        11Gi
Swap:             0B          0B          0B
Wed Sep 23 15:24:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8        

# 2. Authenticate with Hugging Face

We use the Hugging Face access token stored in Google Colab Secrets to authenticate with the Hugging Face Hub.

The token is retrieved securely from the notebook's secret storage rather than being written directly in the notebook. This keeps the credential private while allowing the notebook to access Hugging Face resources.

In [5]:
from google.colab import userdata
from huggingface_hub import login

try:
    # Attempt to retrieve token
    hf_token = userdata.get('HF_TOKEN')

    # Log into the Hugging Face Hub using the retrieved token
    login(hf_token)
    print("Authentication successful.")
except userdata.SecretNotFoundError:
    # If the token isn't found, smoothly continue without crashing
    print("Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.")

Authentication successful.


# 3. Load the InvestIQ Training Dataset

The fine-tuning dataset contains 152 conversation records in JSONL format. JSONL (JSON Lines) stores one JSON object per line, which makes it suitable for conversational fine-tuning datasets.

Each record follows the same conversation structure:

`system → user → assistant`

The assistant response contains the structured JSON output defined for InvestIQ.

Before fine-tuning, we will load the dataset using the Hugging Face datasets library and inspect a sample record before making any transformations.

In [9]:
from datasets import load_dataset

DATA_PATH = "/content/drive/MyDrive/Data Handling and Machine Learning/datasets/jsonl_datasets/InvestIQ_152_dataset.jsonl"

dataset = load_dataset("json", data_files=DATA_PATH, split="train")

print(dataset)
print(f"Number of records: {len(dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 152
})
Number of records: 152


## 3.1. Inspect the Dataset Structure

Before preparing the data for fine-tuning, we inspect one training example.

This allows us to verify that the `system`, `user`, and `assistant` messages were loaded correctly and that the assistant response remains in the expected structured JSON format.

We will inspect the raw conversation before applying the Qwen chat template.

In [10]:
# Inspect the first training example
example = dataset[0]

print(example)

{'messages': [{'role': 'system', 'content': "You are InvestIQ, an AI-assisted investment research assistant. Provide clear, evidence-based analysis using only the information available to you. Explain predictions, interpret financial indicators together, and consider the user's investment goals, risk tolerance, and time horizon. Never invent facts, guarantee returns, or present an investment as a definite buy. If information is insufficient, clearly say so."}, {'role': 'user', 'content': 'Company A has stable revenue, improving operating margins, positive free cash flow growth, and moderate debt-to-equity. Its historical stock volatility is also relatively low. How should I evaluate this profile?'}, {'role': 'assistant', 'content': '{"summary":"The company appears to have a relatively balanced financial profile, with improving profitability, positive cash-generation trends, and moderate leverage. Lower historical volatility may also be relevant for an investor who values stability, alt

# 4. Load the Qwen Tokenizer

Before loading the full language model, we first load the tokenizer for `Qwen/Qwen2.5-1.5B-Instruct`.

A tokenizer converts text into the numerical tokens that the language model can process.

We will also use Qwen's built-in **chat template** to convert our `system → user → assistant` conversation structure into the format expected by the model.

We inspect the tokenizer first so that we can verify the training format before loading the 1.5B parameter model into GPU memory.

In [11]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully


## 4.1. Inspect the Qwen Chat Template

Qwen's chat template defines how conversations are formatted before they are converted into tokens.

Our dataset stores each example as separate `system`, `user`, and `assistant` messages. The tokenizer's `apply_chat_template()` method combines these messages into the format expected by Qwen.

We will apply the template to one example and inspect the resulting text before preparing the full training dataset.

This helps us verify that our dataset is compatible with the model's expected conversational format.

In [12]:
formatted_text = tokenizer.apply_chat_template(
    example["messages"],
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text)

<|im_start|>system
You are InvestIQ, an AI-assisted investment research assistant. Provide clear, evidence-based analysis using only the information available to you. Explain predictions, interpret financial indicators together, and consider the user's investment goals, risk tolerance, and time horizon. Never invent facts, guarantee returns, or present an investment as a definite buy. If information is insufficient, clearly say so.<|im_end|>
<|im_start|>user
Company A has stable revenue, improving operating margins, positive free cash flow growth, and moderate debt-to-equity. Its historical stock volatility is also relatively low. How should I evaluate this profile?<|im_end|>
<|im_start|>assistant
{"summary":"The company appears to have a relatively balanced financial profile, with improving profitability, positive cash-generation trends, and moderate leverage. Lower historical volatility may also be relevant for an investor who values stability, although it does not guarantee lower fu

# 4.2. Load the Qwen Model

We will load the Qwen2.5-1.5B-Instruct model using Hugging Face's `AutoModelForCausalLM` class.

The model is loaded with float16 precision to reduce memory usage and allow it to run efficiently on the GPU.

We also disable the model's cache during training, as caching is not needed for fine-tuning and can use additional memory.

In [13]:
from transformers import AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype=torch.float16
)

model.config.use_cache = False

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# 4.3. Test the Model

We test the loaded Qwen model with a simple user question. The chat template formats the message, and the tokenizer converts it into tensors for the model.

The model then generates a response with a maximum of 50 new tokens. Finally, the generated tokens are decoded back into readable text so we can inspect the model's answer.

In [14]:
messages = [{"role": "user", "content": "Explain photosynthesis to me."}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]

print(
    "STARTING MODEL ANSWER:\n",
    tokenizer.decode(generated_ids, skip_special_tokens=True)
)

STARTING MODEL ANSWER:
 Photosynthesis is the process by which plants and some other organisms use light energy to synthesize food from carbon dioxide and water. It is an essential biological process that provides the primary source of oxygen in our atmosphere and produces organic compounds such as glucose for plant


# 4.4. Preprocess the Dataset

We preprocess each dataset example by applying Qwen's chat template and tokenizing the resulting text. Truncation ensures that sequences that are too long are shortened to fit the model's limits.

The processed dataset is then created using `dataset.map()`.

In [15]:
def preprocess(sample):
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        padding=False
    )
    return tokenized

data = dataset.map(
    preprocess,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/152 [00:00<?, ? examples/s]

# 4.5. Configure LoRA

We configure LoRA (Low-Rank Adaptation) to fine-tune the model efficiently. The configuration specifies the target layers, rank, scaling factor, and bias settings.

The LoRA configuration is then applied to the model using `get_peft_model()`.

In [16]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    bias="none"
)

model = get_peft_model(model, lora_config)

# 4.6. Configure and Start Training

We set up the data collator and training arguments, including the number of epochs, learning rate, batch size, and evaluation strategy.

The dataset is split into training and evaluation sets. Gradient checkpointing is enabled to reduce memory usage during training.

Finally, the `Trainer` trains the model and saves the fine-tuned model and tokenizer to `./my_qwen`.

In [17]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

split = data.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    learning_rate=2e-4,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,

    fp16=True,

    logging_steps=5,
    eval_strategy="epoch",
    lr_scheduler_type="cosine",
    remove_unused_columns=False
)

model.gradient_checkpointing_enable()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()

trainer.save_model("./my_qwen")
tokenizer.save_pretrained("./my_qwen")

Epoch,Training Loss,Validation Loss
1,0.873273,0.755131
2,0.555617,0.705482
3,0.358109,0.760060
4,0.218549,0.922792
5,0.096788,1.036963


('./my_qwen/tokenizer_config.json',
 './my_qwen/chat_template.jinja',
 './my_qwen/tokenizer.json')

# 4.7. Load the Fine-Tuned Model

We reload the base Qwen model and attach the trained LoRA adapter from `./my_qwen`.

The PEFT configuration identifies the original base model, while `PeftModel.from_pretrained()` loads the fine-tuned adapter weights. Caching is then enabled again for inference.

In [18]:
from peft import PeftModel, PeftConfig

path = "./my_qwen"
config = PeftConfig.from_pretrained(path)

base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="cuda",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(base_model, path)

model.config.use_cache = True

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

# 4.8. Test the Fine-Tuned Model

We test the fine-tuned model using the InvestIQ system prompt and a simple user question. The chat template formats the conversation, and the model generates an answer using the trained LoRA adapter.

We test the fine-tuned model with two types of questions. First, we ask an irrelevant question to check whether the model stays within its intended role as an investment assistant. Then, we ask a finance-related question to test its ability to provide relevant investment analysis.

**Irrelevant question:** Tests whether InvestIQ avoids answering questions outside its intended purpose.

In [19]:
messages = [
    {
        "role": "system",
        "content": "You are InvestIQ, an AI-assisted investment research assistant. Provide clear, evidence-based analysis using only the information available to you. Explain predictions, interpret financial indicators together, and consider the user's investment goals, risk tolerance, and time horizon. Never invent facts, guarantee returns, or present an investment as a definite buy. If information is insufficient, clearly say so."
    },
    {
        "role": "user",
        "content": "What is my name?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]

print(
    "FINE-TUNED ANSWER:\n",
    tokenizer.decode(generated_ids, skip_special_tokens=True)
)

FINE-TUNED ANSWER:
 {"summary":"This request cannot be understood because it contains an unrecognized word.","key_factors":[],"risks":[],"probability_interpretation":"","recommendation_context":"I can help explain names, identities, authentication questions, or other text that can be identified as human input.","response_type":"irrelevant_query"}


**Finance question:** Tests whether InvestIQ can provide an appropriate response to an investment-related question.

In [21]:
messages = [
    {
        "role": "system",
        "content": "You are InvestIQ, an AI-assisted investment research assistant. Provide clear, evidence-based analysis using only the information available to you. Explain predictions, interpret financial indicators together, and consider the user's investment goals, risk tolerance, and time horizon. Never invent facts, guarantee returns, or present an investment as a definite buy. If information is insufficient, clearly say so."
    },
    {
        "role": "user",
        "content": "Which financial indicators should I look at when comparing two companies for a long-term investment, and why?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=250,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]

print(
    "FINE-TUNED ANSWER:\n",
    tokenizer.decode(generated_ids, skip_special_tokens=True)
)

FINE-TUNED ANSWER:
 {"summary": "Compare profitability, revenue growth, cash flow, debt-to-equity, historical performance, and volatility. These provide different perspectives on business strength and risk.", "key_factors": ["profitability", "revenue growth", "cash flow", "debt-to-equity", "historical performance", "volatility"], "risks": ["comparison may hide differences", "financial indicators do not measure every risk", "past performance does not guarantee future results"], "probability_interpretation": "", "recommendation_context": "", "response_type": "investment_analysis"}


# 5. Save the Fine-Tuned Model to Google Drive

The trained LoRA adapter and tokenizer are currently stored in the temporary Colab runtime. We copy the saved model to Google Drive so that it remains available after the Colab session ends.

In [22]:
import shutil
import os

SOURCE_PATH = "./my_qwen"

DRIVE_MODEL_PATH = "/content/drive/MyDrive/Data Handling and Machine Learning/week_04/archive/InvestIQ_Qwen_LoRA"

os.makedirs(os.path.dirname(DRIVE_MODEL_PATH), exist_ok=True)

shutil.copytree(
    SOURCE_PATH,
    DRIVE_MODEL_PATH,
    dirs_exist_ok=True
)

print("Model saved to Google Drive:")
print(DRIVE_MODEL_PATH)

Model saved to Google Drive:
/content/drive/MyDrive/Data Handling and Machine Learning/week_04/archive/InvestIQ_Qwen_LoRA
